# Contract Risk Knowledge Base: Cleaning, Chunking, and Chroma Storage

This notebook prepares the two contract reference PDFs for a RAG application. It extracts clean text, creates page- and clause-aware chunks, embeds them with BAAI/bge-m3 model, and stores the chunks and citation metadata in a persistent Chroma database.

NO llm MODEL IS USED YET    

In [3]:
import sys
print(sys.executable)
import importlib.util

print("Python:", sys.executable)
for package in ["chromadb", "pymupdf", "sentence_transformers", "rank_bm25", "openai"]:
    print(package, bool(importlib.util.find_spec(package)))

d:\omar\contractRisk(hackathon)\venv\Scripts\python.exe
Python: d:\omar\contractRisk(hackathon)\venv\Scripts\python.exe
chromadb True
pymupdf True
sentence_transformers True
rank_bm25 True
openai True


In [ ]:
from pathlib import Path
import re
import hashlib
import json

import chromadb
import pymupdf
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() in {"codes", "code"} else Path.cwd().resolve()
KNOWLEDGE_BASE = ROOT / "knowledge_base"
DATABASE_DIR = ROOT / "database" / "chroma_contract_risk"
COLLECTION_NAME = "contract_risk_knowledge_base_v1"

PDF_FILES = [
    KNOWLEDGE_BASE / "Model_services_core_terms.pdf",
    KNOWLEDGE_BASE / "Buyer_guidance.pdf",
]
assert all(path.exists() for path in PDF_FILES), f"Missing source PDF(s): {[p for p in PDF_FILES if not p.exists()]}"

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
CHUNK_CHARS = 3000       # roughly 700–900 English tokens; preserve clause boundaries first
CHUNK_OVERLAP_CHARS = 350
BATCH_SIZE = 32
# TOP_DENSE = 20
# TOP_BM25 = 20
# TOP_RERANK = 8


d:\omar\contractRisk(hackathon)\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Clean and extract the PDFs

Each page is extracted separately so chunks can cite the correct source and page. Repeated headers and footers are removed while the document text is retained.

In [5]:
def clean_page_text(text: str) -> str:
    text = text.replace("\u200b", "").replace("\xa0", " ").replace("\ufeff", "")
    # Normalize spacing while preserving paragraph and page boundaries.
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.splitlines()]
    # Remove repeated document furniture, but retain headings and substantive numbered text.
    kept = []  # Keep substantive text; drop only repeated page furniture.
    for line in lines:
        if not line:
            if kept and kept[-1] != "":
                kept.append("")
            continue
        if re.fullmatch(r"(?:Core Terms|Model Services Contract Guidance)", line, re.I):
            continue
        if re.fullmatch(r"(?:Core Terms – Model Services Contract,? v?2\.2A?|\d{1,3}|Crown copyright(?: 2025)?|\[Subject to contract\])", line, re.I):
            continue
        if re.fullmatch(r"Version 2\.2\(A\) \(Updated September 2025\).*", line, re.I):
            continue
        kept.append(line)
    return "\n".join(kept).strip()


def extract_pages(pdf_path: Path) -> list[dict]:
    # Open one source PDF and attach its filename and one-based page to every text record.
    pdf = pymupdf.open(pdf_path)
    pages = []
    for page_number, page in enumerate(pdf, start=1):
        text = clean_page_text(page.get_text("text", sort=True))
        if text:
            pages.append({"source": pdf_path.name, "page": page_number, "text": text})
    pdf.close()
    return pages

pages = [page for path in PDF_FILES for page in extract_pages(path)]
print(f"Extracted {len(pages)} non-empty pages from {len(PDF_FILES)} PDFs")
print("Page counts:", {p.name: len(pymupdf.open(p)) for p in PDF_FILES})

Extracted 216 non-empty pages from 2 PDFs
Page counts: {'Model_services_core_terms.pdf': 84, 'Buyer_guidance.pdf': 132}


## Create Headers aware chunks

Numbered clauses and headings start new chunks, and a chunk never crosses a PDF page. Long sections are split near paragraph boundaries with a small overlap, keeping citations and surrounding context together.

In [6]:
# Numbering depth 1–3 marks a new retrievable unit (e.g. Clause 10, 10.1, 10.1.1).
# Deeper numbering stays with its parent so chunks retain enough context.
NUMBERED_LINE = re.compile(r"^\s*(?P<num>\d{1,2}(?:\.\d{1,3}){0,2})\.?\s*(?P<rest>.*)$")

TEXTUAL_HEADING = re.compile(
    r"^(?:Section\s+[A-I](?::)?.{0,140}|Annex\s+\d+.{0,140}|Schedule\s+\d+.{0,140}|Contents|FAQs|Introduction to this Document|Contract Selection Guide)$",
    re.I,
)


def heading_for_line(line: str):
    """Recognize non-numbered structural headings; numbered units use NUMBERED_LINE."""
    value = re.sub(r"\s+", " ", line).strip()
    if not value or len(value) > 180:
        return None
    if TEXTUAL_HEADING.fullmatch(value):
        return value
    return None


def _unit_label(number: str, rest: str, parent: str) -> str:
    title = re.sub(r"\s+", " ", rest).strip(" .:–-")
    # Long numbered paragraphs are clause text, not useful titles. Keep the clause id and hierarchy.
    if len(title) > 110:
        title = title[:107].rsplit(" ", 1)[0] + "..."
    prefix = f"{parent} > " if parent and parent != "Page text" else ""
    return f"{prefix}{number}" + (f" {title}" if title else "")


def page_sections(page: dict) -> list[dict]:
    """Split on legal/guidance numbering while preserving every source line and hierarchy."""
    lines = page["text"].splitlines()
    # Some PDF headings put the number and title on separate lines; join them.
    normalized, i = [], 0
    while i < len(lines):
        line = lines[i].strip()
        standalone = re.fullmatch(r"(\d{1,2}(?:\.\d{1,3}){0,2})\.?", line)
        if standalone and i + 1 < len(lines):
            nxt = lines[i + 1].strip()
            if nxt and len(nxt) <= 150 and not NUMBERED_LINE.match(nxt):
                normalized.append(f"{standalone.group(1)} {nxt}")
                i += 2
                continue
        normalized.append(lines[i])
        i += 1

    sections, current_lines = [], []
    parent_heading = "Page text"
    current_label = parent_heading

    def flush():
        body = "\n".join(current_lines).strip()
        if body:
            sections.append({"section": current_label, "text": body})

    for line in normalized:
        stripped = line.strip()
        # Text headings establish context for numbered clauses that follow.
        textual = heading_for_line(stripped)
        numbered = NUMBERED_LINE.match(line)
        if textual and not numbered:
            flush()
            parent_heading = textual
            current_label = textual
            current_lines = [line]
            continue
        if numbered:
            number, rest = numbered.group("num"), numbered.group("rest")
            depth = number.count(".") + 1
            # Start a chunk at substantive numbered headings and clauses.
            if depth <= 3 and rest.strip():
                flush()
                current_label = _unit_label(number, rest, parent_heading)
                current_lines = [line]
                continue
        current_lines.append(line)

    flush()
    return sections


def split_long_text(text: str, limit: int, overlap: int) -> list[str]:
    """Pack paragraphs up to the character budget; overlap only within one legal unit."""
    # Keep paragraph boundaries when possible; split only when a unit is too large.
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    pieces, current = [], ""
    for para in paragraphs:
        if len(para) > limit:
            if current:
                pieces.append(current.strip())
                current = ""
            start = 0
            while start < len(para):
                end = min(start + limit, len(para))
                if end < len(para):
                    boundary = max(para.rfind(". ", start, end), para.rfind("; ", start, end), para.rfind(" ", start, end))
                    if boundary > start + int(limit * 0.65):
                        end = boundary + 1
                pieces.append(para[start:end].strip())
                if end >= len(para):
                    break
                start = max(start + 1, end - overlap)
            continue
        candidate = f"{current}\n\n{para}" if current else para
        if len(candidate) <= limit:
            current = candidate
        else:
            pieces.append(current.strip())
            current = f"{current[-overlap:]}\n\n{para}".strip() if current else para
    if current:
        pieces.append(current.strip())
    return pieces


def make_chunks(pages: list[dict]) -> list[dict]:
    chunks = []
    for page in pages:
        # Page boundaries are deliberate: each resulting chunk has an exact page citation.
        for section in page_sections(page):
            for part_no, part in enumerate(split_long_text(section["text"], CHUNK_CHARS, CHUNK_OVERLAP_CHARS), start=1):
                content = f"Document: {page['source']}\nPage: {page['page']}\nSection: {section['section']}\n\n{part}"
                digest = hashlib.sha256(f"{page['source']}|{page['page']}|{section['section']}|{part_no}|{part}".encode("utf-8")).hexdigest()[:24]
                clause_match = re.search(r"(?:Clause )?(\d{1,2}(?:\.\d{1,3}){0,2})", section["section"])
                chunks.append({
                    "id": digest,
                    "text": content,
                    "source": page["source"],
                    "page": page["page"],
                    "section": section["section"],
                    "clause": clause_match.group(1) if clause_match else "",
                })
    return chunks

chunks = make_chunks(pages)
print(f"Created {len(chunks)} section-aware chunks")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2)[:1200])

Created 1459 section-aware chunks
{
  "id": "d83181f4cc094b8856203c98",
  "text": "Document: Model_services_core_terms.pdf\nPage: 1\nSection: Page text\n\nGLD Standard Form Document\n\nCategory Model Contract – Provision of Services\nTitle Model Services Contract\nMaturity Level Medium\n\nNotes:",
  "source": "Model_services_core_terms.pdf",
  "page": 1,
  "section": "Page text",
  "clause": ""
}


## Embed chunks and store them in Chroma


In [7]:
# Load the embedding model used to turn each text chunk into a vector.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)


def embed_texts(texts: list[str]) -> list[list[float]]:
    # Normalize vectors so cosine similarity is appropriate for semantic search.
    vectors = embedding_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    return vectors.tolist()

# PersistentClient saves Chroma data in the project's database folder.
chroma_client = chromadb.PersistentClient(path=str(DATABASE_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine", "embedding_model": EMBEDDING_MODEL_NAME},
)

#Refresh only these source documents so edits/reingestion don't leave stale page chunks.
source_names = [p.name for p in PDF_FILES]
# Refresh only these two PDFs; unrelated records in this collection are kept.
collection.delete(where={"source": {"$in": source_names}})

# Add vectors in batches and keep source/page/section metadata for future citations.
for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start : start + BATCH_SIZE]
    vectors = embed_texts([item["text"] for item in batch])
    collection.upsert(
        ids=[item["id"] for item in batch],
        documents=[item["text"] for item in batch],
        embeddings=vectors,
        metadatas=[{"source": item["source"], "page": item["page"], "section": item["section"], "clause": item["clause"] or "none"} for item in batch],
    )
print(f"Indexed {collection.count()} chunks in {DATABASE_DIR}")

d:\omar\contractRisk(hackathon)\venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Omar\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\omar\contractRisk(hackathon)\venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hu

Indexed 1459 chunks in D:\omar\contractRisk(hackathon)\database\chroma_contract_risk


### database inspection
cecking our chunks and chunk split and samples

In [8]:
from pathlib import Path
import chromadb

# Find the project root whether Jupyter started in "codes" or the root folder.
root = Path.cwd().resolve()
if root.name.lower() in {"codes", "code"}:
    root = root.parent

db_path = root / "database" / "chroma_contract_risk"
client = chromadb.PersistentClient(path=str(db_path))
collection = client.get_collection("contract_risk_knowledge_base_v1")

print("Database:", db_path)
print("Stored chunks:", collection.count())

sample = collection.get(
    limit=5,
    include=["documents", "metadatas"],
)

for chunk_id, document, metadata in zip(
    sample["ids"], sample["documents"], sample["metadatas"]
):
    print(f"\nID: {chunk_id}")
    print(f"Source: {metadata['source']} | Page: {metadata['page']} | Section: {metadata['section']}")
    print(document[:500], "...")

Database: D:\omar\contractRisk(hackathon)\database\chroma_contract_risk
Stored chunks: 1459

ID: d83181f4cc094b8856203c98
Source: Model_services_core_terms.pdf | Page: 1 | Section: Page text
Document: Model_services_core_terms.pdf
Page: 1
Section: Page text

GLD Standard Form Document

Category Model Contract – Provision of Services
Title Model Services Contract
Maturity Level Medium

Notes: ...

ID: ffed1696038a2a1c0e951cab
Source: Model_services_core_terms.pdf | Page: 1 | Section: 1 This standard form document is intended for use by lawyers
Document: Model_services_core_terms.pdf
Page: 1
Section: 1 This standard form document is intended for use by lawyers

1. This standard form document is intended for use by lawyers. ...

ID: f1fb4034b21c9452c39cff58
Source: Model_services_core_terms.pdf | Page: 1 | Section: 2 Use of this document is strongly recommended because it is up to date and reflects
Document: Model_services_core_terms.pdf
Page: 1
Section: 2 Use of this document is strongly

In [9]:
from pathlib import Path
from collections import Counter
import chromadb

# Find the project root whether Jupyter started in "codes" or the root folder.
root = Path.cwd().resolve()
if root.name.lower() in {"codes", "code"}:
    root = root.parent

db_path = root / "database" / "chroma_contract_risk"
target_name = "contract_risk_knowledge_base_v1"
expected_sources = {"Model_services_core_terms.pdf", "Buyer_guidance.pdf"}

client = chromadb.PersistentClient(path=str(db_path))

# Show what collections actually exist in this database.
listed = client.list_collections()
collection_names = [
    item if isinstance(item, str) else item.name
    for item in listed
]
print("Database path:", db_path)
print("Collections found:", collection_names)

if target_name not in collection_names:
    print(f"\nNOT FOUND: {target_name}")
else:
    collection = client.get_collection(target_name)
    total = collection.count()
    print(f"\nCollection: {collection.name}")
    print("Total chunks:", total)
    print("Collection settings:", collection.metadata)

    # Read metadata only for the corpus-wide summary; no vectors are loaded here.
    records = collection.get(include=["metadatas"])
    metadatas = records["metadatas"] or []

    counts_by_source = Counter(m.get("source", "missing source") for m in metadatas)
    print("\nChunks per source:")
    for source, count in sorted(counts_by_source.items()):
        print(f"  {source}: {count}")

    print("\nCoverage per source:")
    for source in sorted(expected_sources):
        source_rows = [m for m in metadatas if m.get("source") == source]
        pages = sorted({m.get("page") for m in source_rows if m.get("page") is not None})
        sections = {m.get("section") for m in source_rows if m.get("section")}
        clauses = {m.get("clause") for m in source_rows if m.get("clause") not in (None, "", "none")}
        print(
            f"  {source}: {len(source_rows)} chunks, "
            f"{len(pages)} distinct pages, {len(sections)} sections, "
            f"{len(clauses)} clause IDs"
        )
        if pages:
            print(f"    Page range: {min(pages)}–{max(pages)}")

    missing = expected_sources - set(counts_by_source)
    if total and not missing:
        print("\nPASS: both expected PDFs are present in the collection.")
    else:
        print("\nCHECK: collection is empty or these sources are missing:", sorted(missing))

    # Inspect one stored chunk from each source, including its citation metadata.
    print("\nExample stored chunks:")
    for source in sorted(expected_sources & set(counts_by_source)):
        sample = collection.get(
            where={"source": source},
            limit=1,
            include=["documents", "metadatas"],
        )
        if sample["ids"]:
            metadata = sample["metadatas"][0]
            print(
                f"\n  {source} | page {metadata.get('page')} | "
                f"{metadata.get('section')}"
            )
            print(sample["documents"][0][:400], "...")

    # Check vector dimensions using only one stored embedding.
    if total:
        vector_record = collection.get(limit=1, include=["embeddings"])
        vector = vector_record["embeddings"][0]
        print("\nEmbedding dimensions:", len(vector))

Database path: D:\omar\contractRisk(hackathon)\database\chroma_contract_risk
Collections found: ['contract_risk_knowledge_base_v1']

Collection: contract_risk_knowledge_base_v1
Total chunks: 1459
Collection settings: {'hnsw:space': 'cosine', 'embedding_model': 'BAAI/bge-m3'}

Chunks per source:
  Buyer_guidance.pdf: 494
  Model_services_core_terms.pdf: 965

Coverage per source:
  Buyer_guidance.pdf: 494 chunks, 132 distinct pages, 321 sections, 125 clause IDs
    Page range: 1–132
  Model_services_core_terms.pdf: 965 chunks, 84 distinct pages, 918 sections, 691 clause IDs
    Page range: 1–84

PASS: both expected PDFs are present in the collection.

Example stored chunks:

  Buyer_guidance.pdf | page 1 | Page text
Document: Buyer_guidance.pdf
Page: 1
Section: Page text

Model Services Contract -
Guidance for Authorities

For use under the Procurement Act 2023

Version 2.2(A) (September 2025)

copyright ...

  Model_services_core_terms.pdf | page 1 | Page text
Document: Model_services_c